In [10]:
import shutil
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import os
from pathlib import Path
from tqdm import tqdm
from isds_tool.PS_data.zip_tools import uzip_file

In [12]:
input_image_dir = r'/localnvme/data/billboard/fused_data/data7961_mseg_c6_1015/images'
input_label_dir = r'/localnvme/data/billboard/fused_data/data7961_mseg_c6_1015/labels'
output_label_dir = r'/localnvme/data/added_data/check1020/data_mseg_c6/labels'
check_image_dir = r'/localnvme/data/added_data/check1020/check_dir_re'
broken_dir_high = 'broken_high_1020'
broken_dir_medium = 'broken_medium_1020'
no_dir_list = [
    'no_1020',
    'crack_high_1020',
    'crack_medium_1020',
    'hole_high_1020',
    'hole_medium_1020',
]
os.makedirs(output_label_dir, exist_ok=True)

In [13]:
# file_list = os.listdir(check_image_dir)
# for file_name in file_list:
#     if file_name.endswith('.zip'):
#         file_path = os.path.join(check_image_dir, file_name)
#         uzip_file(file_path, file_path.replace('.zip', ''))

In [14]:
def get_risk_refine_broken(check_image_dir, obj_image_name):
    broken_high_path = os.path.join(check_image_dir, broken_dir_high, obj_image_name)
    if os.path.exists(broken_high_path):
        return 2
    broken_medium_path = os.path.join(check_image_dir, broken_dir_medium, obj_image_name)
    if os.path.exists(broken_medium_path):
        return 1
    for no_dir_name in no_dir_list:
        no_dir_path = os.path.join(check_image_dir, no_dir_name, obj_image_name)
        if os.path.exists(no_dir_path):
            return 0
    return -1

def risk_refine_broken(input_image_dir, check_image_dir, input_gt_dir, output_gt_dir):
    os.makedirs(output_gt_dir, exist_ok=True)
    image_list = os.listdir(input_image_dir)
    broken_high_src_count, broken_medium_src_count, broken_no_src_count = 0, 0, 0
    broken_high_dst_count, broken_medium_dst_count, broken_no_dst_count = 0, 0, 0
    not_find_list = []
    for image_name in tqdm(image_list):
        keep = False
        label_name = Path(image_name).stem + '.txt'
        input_gt_path = os.path.join(input_gt_dir, label_name)
        output_gt_path = os.path.join(output_gt_dir, label_name)
        with open(input_gt_path, 'r') as fi:
            data = fi.readlines()
            new_data = []
            for id_line, line in enumerate(data):
                parts = line.strip().split(' ')
                category = int(parts[0])
                att_len = int(parts[1])
                atts = list(map(int, parts[2:2 + att_len]))
                polygons = list(map(float, parts[2 + att_len:]))

                risk_broken = atts[1]
                if risk_broken > 0:
                    keep = True
                    if risk_broken == 1:
                        broken_medium_src_count += 1
                    else:
                        broken_high_src_count += 1
                    obj_image_name = Path(image_name).stem + f'_{id_line}' + Path(image_name).suffix
                    obj_result = get_risk_refine_broken(check_image_dir, obj_image_name)
                    if obj_result == 0:
                        broken_no_dst_count += 1
                    elif obj_result == 1:
                        broken_medium_dst_count += 1
                    elif obj_result == 2:
                        broken_high_dst_count += 1
                    else:
                        print(f'{obj_image_name} cannot find!')
                        not_find_list.append(obj_image_name)
                        obj_result = risk_broken
                    atts[1] = obj_result
                info = [category, att_len] + atts + polygons
                new_line = ' '.join(map(str, info)) +'\n'
                new_data.append(new_line)
            if keep:
                with open(output_gt_path, 'w') as fo:
                    fo.writelines(new_data)
    print(f'{len(not_find_list)} not find')
    return not_find_list

In [15]:
not_find_list = risk_refine_broken(input_image_dir, check_image_dir, input_label_dir, output_label_dir)

100%|██████████| 7872/7872 [00:03<00:00, 2352.93it/s]

0 not find


In [16]:
print(not_find_list)

[]


In [17]:
input_dir1 = r'/localnvme/data/billboard/fused_data/data7961_mseg_c6_1015/images_crop/broken/high'
input_dir2 = r'/localnvme/data/billboard/fused_data/data7961_mseg_c6_1015/images_crop/broken/medium'
output_dir = r'/localnvme/data/added_data/check1020/check_dir_re_not_find'

In [18]:
not_find_again = []
os.makedirs(output_dir, exist_ok=True)
for file_name in tqdm(not_find_list):
    input_path1 = os.path.join(input_dir1, file_name)
    input_path2 = os.path.join(input_dir2, file_name)
    output_path = os.path.join(output_dir, file_name)
    if os.path.exists(input_path1):
        shutil.copy(input_path1, output_path)
    elif os.path.exists(input_path2):
        shutil.copy(input_path2, output_path)
    else:
        not_find_again.append(file_name)
print(f'{len(not_find_again)} not find')

0it [00:00, ?it/s]

0 not find
